# Tools and Function Registration

Notebook **06** equips **`AssistantAgent`** with **function tools** — Python callables the model can invoke to fetch Notes API facts, search documentation chunks **c1–c5**, and perform arithmetic. AutoGen 0.4+ infers **tool schemas from type hints** and supports **async** tool functions with **parallel tool calls**.

This notebook covers **`search_docs`**, **`get_note`**, **`add`**, a **`TOOLS`** registry, **tool error handling**, **`reflect_on_tool_use`**, and a comparison to LangChain **`@tool`** (**01. LangChain/12**).

**Prerequisites:** **03. AssistantAgent and ConversableAgent**, **02. Setup and the AgentChat API**.

**What you'll learn:**

- How to register async Python functions as tools on **`AssistantAgent`**
- How type hints become JSON schemas for the model
- How parallel tool calls and error strings keep the agent loop stable
- How AutoGen tools compare to LangGraph **`ToolNode`** (**02. LangGraph/07**)

In [ ]:
OPENAI_API_KEY = "sk-proj-placeholder-replace-with-your-real-key"

try:
    import autogen_agentchat
    print("autogen-agentchat:", getattr(autogen_agentchat, "__version__", "installed"))
except ImportError:
    print("autogen-agentchat: pip install autogen-agentchat autogen-ext[openai]")

try:
    import autogen_ext
    print("autogen-ext:", getattr(autogen_ext, "__version__", "installed"))
except ImportError:
    print("autogen-ext: pip install autogen-ext[openai]")

---

## 1. Why Function Tools?

Without tools, the model guesses facts about your Notes API. Tools ground answers in **retrievable data**:

| Approach | Strength | Weakness |
|----------|----------|----------|
| Prompt-only | Simple | Hallucinates corpus details |
| RAG pipeline | Great for large corpora | More infrastructure |
| **Function tools** | Precise, testable lookups | Model must choose correctly |

AutoGen passes tools to the model client; the agent runtime executes calls and feeds results back — similar to LangGraph's ReAct loop (**02. LangGraph/06**).

---

## 2. Notes API Tool Corpus

Same teaching corpus as **05** and the LangGraph track — chunks **c1–c5** plus note ids **n1–n4**:

In [ ]:
NOTES = {
    "n1": "The Notes API is built with FastAPI. Routes live under /notes.",
    "n2": "Run alembic upgrade head after pulling database migrations.",
    "n3": "JWT bearer tokens use Authorization: Bearer header.",
    "n4": "Pytest fixtures for DB setup belong in conftest.py.",
}

DOC_CHUNKS = [
    {"id": "c1", "text": NOTES["n1"]},
    {"id": "c2", "text": NOTES["n2"]},
    {"id": "c3", "text": NOTES["n3"]},
    {"id": "c4", "text": NOTES["n4"]},
    {"id": "c5", "text": "Use httpx.AsyncClient in FastAPI tests for async endpoints."},
]

print("corpus:", [c["id"] for c in DOC_CHUNKS])

---

## 3. Async Tool Functions

Define tools as **`async def`** when they perform I/O (DB, HTTP, search). AutoGen awaits them automatically:

In [ ]:
import asyncio
from typing import Any


async def search_docs(query: str) -> str:
    """Keyword search over documentation chunks c1-c5."""
    await asyncio.sleep(0.01)  # simulate I/O
    tokens = set(query.lower().split())
    hits = []
    for chunk in DOC_CHUNKS:
        if tokens & set(chunk["text"].lower().split()):
            hits.append(f"[{chunk['id']}] {chunk['text']}")
    return "\n".join(hits) if hits else "No matching chunks."


async def get_note(note_id: str) -> str:
    """Fetch a note by id (n1, n2, n3, n4)."""
    await asyncio.sleep(0.01)
    return NOTES.get(note_id, f"Unknown note id: {note_id}")


async def add(a: int, b: int) -> int:
    """Add two integers."""
    return a + b


print("tools defined:", search_docs.__name__, get_note.__name__, add.__name__)

---

## 4. Tool Schema from Type Hints

AutoGen (via the model client) derives JSON Schema from:

- **Function name** → tool name
- **Docstring** → tool description
- **Type hints** → parameter types and required fields

No separate schema file is needed for simple functions — contrast with manual **`StructuredTool`** in LangChain.

In [ ]:
import inspect

for fn in (search_docs, get_note, add):
    sig = inspect.signature(fn)
    print(f"{fn.__name__}{sig} -> {sig.return_annotation}")
    print("  doc:", (fn.__doc__ or "").strip())

---

## 5. `TOOLS` Registry

A name → callable registry mirrors **`TOOLS_BY_NAME`** in **02. LangGraph/07** — useful for tests and custom dispatch:

In [ ]:
TOOLS: dict[str, Any] = {
    "search_docs": search_docs,
    "get_note": get_note,
    "add": add,
}

print("registered:", list(TOOLS))

---

## 6. Model Client and `AssistantAgent`

Create the model client (**02**) and attach the tool list to **`AssistantAgent`**:

In [ ]:
from autogen_ext.models.openai import OpenAIChatCompletionClient
from autogen_agentchat.agents import AssistantAgent

model_client = OpenAIChatCompletionClient(
    model="gpt-4o-mini",
    api_key=OPENAI_API_KEY,
    temperature=0,
)

notes_agent = AssistantAgent(
    name="notes_assistant",
    model_client=model_client,
    tools=[search_docs, get_note, add],
    system_message=(
        "You are a Notes API assistant. Use search_docs and get_note before "
        "answering factual questions. Use add for arithmetic. Cite chunk ids like [c2]."
    ),
    reflect_on_tool_use=True,
)

print("agent:", notes_agent.name)

---

## 7. `reflect_on_tool_use`

When **`reflect_on_tool_use=True`**, the agent asks the model to synthesize a natural-language answer after tool results arrive — not just raw JSON. This mirrors LangGraph's model node after **`ToolNode`** (**02. LangGraph/07**).

---

## 8. Single-Tool Demo — `search_docs`

Ask a factual question that requires searching chunk **c2** (Alembic migrations):

In [ ]:
from autogen_agentchat.messages import TextMessage

single_result = await notes_agent.run(
    task="Which doc chunk mentions alembic upgrade head? Use search_docs."
)

for msg in single_result.messages:
    if isinstance(msg, TextMessage) and msg.source == "notes_assistant":
        print(msg.content[:300])

---

## 9. Parallel Tool Calls

Modern models may request **multiple tools in one turn** (e.g. **`search_docs`** + **`get_note`** + **`add`**). AutoGen executes them and returns all results before the next model call — like **`ToolNode`** parallel dispatch (**02. LangGraph/07**).

In [ ]:
parallel_result = await notes_agent.run(
    task=(
        "Find JWT auth docs and the n3 note, then add 7 and 5. "
        "Use tools for each step."
    )
)

tool_msgs = [m for m in parallel_result.messages if m.type == "ToolCallSummaryMessage"]
print("tool summary messages:", len(tool_msgs))
print("final:", parallel_result.messages[-1].content[:200] if parallel_result.messages else "")

---

## 10. Tool Error Handling

Tools should return **error strings**, not raise, when possible — so the model can retry. Demonstrate with a strict **`get_note`** wrapper:

In [ ]:
async def get_note_strict(note_id: str) -> str:
    """Fetch a note by id; returns error text instead of raising."""
    if note_id not in NOTES:
        return f"Tool error: unknown note_id '{note_id}'. Valid: {', '.join(NOTES)}"
    return NOTES[note_id]


strict_agent = AssistantAgent(
    name="strict_notes",
    model_client=model_client,
    tools=[search_docs, get_note_strict, add],
    system_message="Use tools. If a tool returns 'Tool error', fix your args and retry.",
    reflect_on_tool_use=True,
)

err_result = await strict_agent.run(task="Call get_note_strict with note_id=n99.")
print(err_result.messages[-1].content[:250])

---

## 11. Manual Registry Dispatch (Teaching Pattern)

The same pattern as **`execute_tool_calls`** in **02. LangGraph/07** — map tool name to callable:

In [ ]:
async def dispatch_tool(name: str, args: dict[str, Any]) -> str:
    fn = TOOLS.get(name)
    if fn is None:
        return f"Unknown tool: {name}"
    try:
        result = fn(**args)
        if asyncio.iscoroutine(result):
            result = await result
        return str(result)
    except Exception as exc:
        return f"Tool error: {exc}"


demo = await dispatch_tool("search_docs", {"query": "jwt bearer"})
print(demo)
demo2 = await dispatch_tool("add", {"a": 3, "b": 9})
print(demo2)

---

## 12. Compare to LangChain `@tool` (**01. LangChain/12**)

| Aspect | LangChain `@tool` | AutoGen function tools |
|--------|-------------------|------------------------|
| Registration | Decorator → **`StructuredTool`** | Plain async/sync functions in list |
| Schema | Inferred from hints + docstring | Same — passed to model client |
| Execution | **`ToolNode`** / manual invoke | Agent runtime handles calls |
| Parallel calls | **`ToolNode`** batches | Agent runtime batches |
| Ecosystem | Rich tool adapters | Bring your own callables |

Both approaches keep **one tool message per tool call** so the model correlates observations.

In [ ]:
# LangChain equivalent (reference only - requires langchain-core)
LANGCHAIN_STYLE = """
from langchain_core.tools import tool

@tool
async def search_docs(query: str) -> str:
    \"\"\"Keyword search over documentation chunks c1-c5.\"\"\"
    ...

tools = [search_docs, get_note, add]
llm_with_tools = llm.bind_tools(tools)
"""
print("See 01. LangChain/12 and 02. LangGraph/07 for the LangChain execution path.")

---

## 13. `run_stream` with Tools

Stream intermediate tool events for debugging (**15**):

In [ ]:
from autogen_agentchat.base import TaskResult

await notes_agent.reset()
async for event in notes_agent.run_stream(task="What file holds pytest DB fixtures? Use search_docs."):
    if isinstance(event, TaskResult):
        print("stop:", event.stop_reason)
    elif hasattr(event, "content") and event.content:
        print(f"[{getattr(event, 'source', type(event).__name__)}]", str(event.content)[:100])

---

## 14. Extending the Toolset

When adding tools, update **three** places — same rule as LangGraph (**02. LangGraph/07**):

1. Define the function with type hints + docstring
2. Add to **`TOOLS`** registry
3. Pass updated list to **`AssistantAgent(tools=[...])`**

In [ ]:
async def list_note_ids() -> str:
    """Return all available note ids."""
    return ", ".join(sorted(NOTES))


TOOLS["list_note_ids"] = list_note_ids
extended_tools = [search_docs, get_note, add, list_note_ids]

extended_agent = AssistantAgent(
    name="extended_notes",
    model_client=model_client,
    tools=extended_tools,
    system_message="Use list_note_ids when the user asks what notes exist.",
    reflect_on_tool_use=True,
)

ids_result = await extended_agent.run(task="What note ids are available?")
print(ids_result.messages[-1].content[:200])

---

## 15. Design Guidelines

| Guideline | Rationale |
|-----------|-----------|
| **Async tools for I/O** | Don't block the event loop in FastAPI services |
| **Docstrings are the schema description** | Model routing depends on clear tool docs |
| **Return errors as strings** | Lets the model self-correct |
| **Keep `TOOLS` in sync** | Tests and manual dispatch need the registry |
| **`reflect_on_tool_use=True`** | Users get prose, not raw tool JSON |
| **Start with 3–5 tools** | Too many tools confuse routing — split agents (**08**) |

---

## 16. Common Mistakes

| Mistake | Consequence | Fix |
|---------|-------------|-----|
| Missing type hints | Weak or wrong JSON schema | Annotate all parameters |
| Vague docstrings | Model picks wrong tool | State what the tool returns |
| Raising in tools | Agent run may fail | Return **`Tool error: ...`** strings |
| Tools not in agent list | Model calls missing tools | Pass full list to **`AssistantAgent`** |
| Sync blocking I/O in tools | Slow parallel execution | Use **`async def`** |
| Forgetting registry update | Tests dispatch stale tools | Update **`TOOLS`** dict |

---

## 17. Summary

**`AssistantAgent`** tools are plain Python functions — async or sync — with type hints and docstrings forming the model's JSON schema. The agent runtime handles **parallel tool calls**, feeds results back, and optionally **reflects** with **`reflect_on_tool_use`**.

Key takeaways:

- **`TOOLS`** registry mirrors LangGraph's **`TOOLS_BY_NAME`** pattern.
- Tool **errors as strings** keep the loop recoverable — same philosophy as **`ToolNode`** error handling.
- For large tool sets, split into specialist agents (**08**) instead of one mega-agent.

Demonstrations registered **`search_docs`**, **`get_note`**, and **`add`**, ran single and multi-tool queries, handled bad note ids, and compared to LangChain **`@tool`**.

Next: **07. Code Execution and Sandboxing** — run generated code safely with **`CodeExecutorAgent`** and approval guardrails.